In [2]:
#=====================================================================================
# Welcome to your new notebook
# NOTEBOOK: Ingest Weather Data from OpenWeatherMap API
# Layer: Bronze
# Pattern: Daily full refresh per configured cities
#========================================================================================

StatementMeta(, e28ae8b6-c619-43af-996a-bdb4379ea26c, 4, Finished, Available, Finished, False)

In [3]:
#Import libraries

import requests
import json
from datetime import datetime, timezone

StatementMeta(, e28ae8b6-c619-43af-996a-bdb4379ea26c, 5, Finished, Available, Finished, False)

In [4]:
#API configuration
import requests

api_key = "ff263f0efd75ab8cdceb8dcb595fb2ea"

url = "https://api.openweathermap.org/data/2.5/forecast"

params = {
    "q": "mumbai",
    "appid": api_key,
    "units": "metric",
    "cnt": 40
}

#Call API
response = requests.get(url, params=params, timeout=30)

print("Status Code:", response.status_code)
print("Response Text:", response.text[:500])

StatementMeta(, e28ae8b6-c619-43af-996a-bdb4379ea26c, 6, Finished, Available, Finished, False)

Status Code: 200
Response Text: {"cod":"200","message":0,"cnt":40,"list":[{"dt":1781643600,"main":{"temp":30.28,"feels_like":34.98,"temp_min":30.28,"temp_max":30.28,"pressure":1008,"sea_level":1008,"grnd_level":1007,"humidity":67,"temp_kf":0},"weather":[{"id":800,"main":"Clear","description":"clear sky","icon":"01n"}],"clouds":{"all":9},"wind":{"speed":4.97,"deg":264,"gust":4.97},"visibility":10000,"pop":0,"sys":{"pod":"n"},"dt_txt":"2026-06-16 21:00:00"},{"dt":1781654400,"main":{"temp":30.2,"feels_like":34.55,"temp_min":30.05


In [5]:
weather_data =  response.json()
print(weather_data)

StatementMeta(, e28ae8b6-c619-43af-996a-bdb4379ea26c, 7, Finished, Available, Finished, False)

In [7]:
print(weather_data["cod"])
print(weather_data["cnt"])
print(len(weather_data["list"]))

StatementMeta(, e28ae8b6-c619-43af-996a-bdb4379ea26c, 9, Finished, Available, Finished, False)

200
40
40


In [8]:
# this code as saving the weather API response permanently to a file.

import os 
import json
from datetime import datetime,timezone 
folder_path = "/lakehouse/default/Files/weather_data/raw_json"

os.makedirs(folder_path,exist_ok=True)

run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

#This creates final file path.
raw_file_path = f"{folder_path}/weather_mumbai_{run_timestamp}.json"

with open(raw_file_path, "w") as file:
    json.dump(weather_data, file, indent=4)

print(f"Raw JSON saved successfully at: {raw_file_path}")

StatementMeta(, e28ae8b6-c619-43af-996a-bdb4379ea26c, 10, Finished, Available, Finished, False)

Raw JSON saved successfully at: /lakehouse/default/Files/weather_data/raw_json/weather_mumbai_20260616_204145.json


In [18]:
from pyspark.sql.types import *
from datetime import datetime, timezone

StatementMeta(, e28ae8b6-c619-43af-996a-bdb4379ea26c, 20, Finished, Available, Finished, False)

In [20]:

# from pyspark.sql.types import StructField,StructType,StringType,DoubleType,IntegralType


# ── Step 1: read city info (once, outside the loop) ──   select city,name

city_name = weather_data["city"]["name"]
country_name = weather_data["city"]["country"]

# Step 2: define schema
# 
schema = StructType([
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
    StructField("forecast_time", StringType(), True),
    StructField("temp_c", StringType(),True),
    StructField("humidity_pct", IntegerType(), True),
    StructField("pressure_hpa", IntegerType(), True),
    StructField("weather_main", StringType(), True),
    StructField("weather_desc", StringType(), True),
    StructField("wind_speed_ms", DoubleType(), True),
    StructField("cloudiness_pct", IntegerType(), True),
    StructField("rain_3h_mm", DoubleType(), True),
    StructField("ingested_at", StringType(), True)

])
# Step 3: create records with fixed types
  # ________loop each 3-hour slot ────────────────────

records =[]

for slot in weather_data["list"]:
    records.append({
        "city": str(city_name),
        "country": str(country_name),
        "forecast_time": str(slot["dt_txt"]),
        "temp_c": float(slot["main"]["temp"]),
        "humidity_pct": int(slot["main"]["humidity"]),
        "pressure_hpa": int(slot["main"]["pressure"]),
        "weather_main": str(slot["weather"][0]["main"]),
        "weather_desc": str(slot["weather"][0]["description"]),
        "wind_speed_ms": float(slot["wind"]["speed"]),
        "cloudiness_pct": int(slot["clouds"]["all"]),
        "rain_3h_mm": float(slot.get("rain", {}).get("3h", 0.0)),
        "ingested_at": datetime.now(timezone.utc).isoformat()
    })

# ── Step 4: Spark DataFrame ───────────────────────────

df_weather = spark.createDataFrame(records, schema = schema)
display(df_weather)




StatementMeta(, e28ae8b6-c619-43af-996a-bdb4379ea26c, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a86a5128-5568-4985-bbf1-c7c395898940)

In [23]:
# ── Step 5: Write to Bronze Lakehouse (Delta Table) ─────────────

df_weather.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", True) \
.saveAsTable("bronze_weather")         # saves directly into your attached Lakehouse

print(f"[INFO] Written to bronze_weather | Rows: {df_weather.count()}")

StatementMeta(, e28ae8b6-c619-43af-996a-bdb4379ea26c, 25, Finished, Available, Finished, False)

[INFO] Written to bronze_weather | Rows: 40
